# Входы и выходы слоёв Keras — версия 2.0

    **Цель:** Проследить изменение формы тензора при разных операциях.

    **Алгоритм:** Dense, Flatten, Reshape, Concatenate, свёртки, pooling, Embedding и состояния LSTM.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('keras-layer-input-output')


## Настройка


In [ ]:
from ml_lab.runtime import tensorflow_setup
tf = tensorflow_setup(cfg)
import numpy as np
from tensorflow.keras import layers, Model


## Dense, Flatten, Reshape и Concatenate
Формы Input всегда заданы кортежами.


In [ ]:
inp = layers.Input(shape=(64,))
dense = Model(inp, layers.Dense(10)(layers.Dense(200, activation='relu')(inp)))
assert dense(np.zeros((2, 64), dtype='float32')).shape == (2, 10)
inp = layers.Input(shape=(64, 64))
hidden = layers.Dense(100)(inp)
flatten = Model(inp, layers.Flatten()(hidden))
reshape = Model(inp, layers.Reshape((6400,))(hidden))
print('Dense:', dense.output_shape, 'Flatten:', flatten.output_shape, 'Reshape:', reshape.output_shape)
a, b = layers.Input(shape=(32, 32, 3)), layers.Input(shape=(32, 32, 3))
joined = Model([a, b], layers.Concatenate(axis=-1)([a, b]))
print('Concatenate:', joined.output_shape)
example = np.array([[[2, 4, 6, 8], [1, 3, 5, 7]]], dtype='float32')
weights = np.array([[.3, .5]] * 4, dtype='float32')
print('Matrix multiply:', (example @ weights).shape)


## Conv2D, MaxPooling, увеличение разрешения


In [ ]:
for padding, stride, dilation in [('same', 1, 1), ('valid', 1, 1), ('same', 2, 1), ('valid', 1, 2)]:
    inp = layers.Input(shape=(32, 32, 3))
    conv = Model(inp, layers.Conv2D(32, 3, padding=padding, strides=stride, dilation_rate=dilation)(inp))
    print(padding, stride, dilation, conv.output_shape)
for operation in [layers.MaxPooling2D(2), layers.UpSampling2D(2), layers.Conv2DTranspose(1, 3, strides=2, padding='same')]:
    inp = layers.Input(shape=(8, 8, 1))
    demo = Model(inp, operation(inp))
    print(type(operation).__name__, demo.output_shape)
    assert demo(np.zeros((1, 8, 8, 1), dtype='float32')).shape[0] == 1


## Embedding, Conv1D и LSTM
Индексы слов — целые числа; batch и длина последовательности заданы явно.


In [ ]:
tokens = np.arange(10).reshape(1, 10).astype('int32')
inp = layers.Input(shape=(10,), dtype='int32')
embedded = layers.Embedding(1000, 20)(inp)
print('Embedding:', Model(inp, embedded)(tokens).shape)
print('Conv1D:', Model(inp, layers.Conv1D(32, 5, padding='same')(embedded))(tokens).shape)
for sequence in [False, True]:
    output, h, c = layers.LSTM(64, return_sequences=sequence, return_state=True)(embedded)
    model = Model(inp, [output, h, c])
    print('return_sequences=', sequence, [v.shape for v in model(tokens)])


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Рассчитайте формы выходов до запуска.
2. Сравните same и valid.
3. Объясните разницу return_sequences и return_state.
4. Почему (64,) — кортеж, а (64) — число?

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
